# Crystal symmetry

We use the `SpaceGroup` class to look up symmetry operators for the 230
crystallographic space groups, expand an asymmetric unit into the full unit
cell, and check which reflections are systematically absent. We work through
three structures (NaCl, silicon, corundum) and finish with a table of allowed
reflections for each, ready for intensity calculations.

In [1]:
import numpy as np

from diffraction import Crystal, SpaceGroup

## Space group lookup

A `SpaceGroup` can be constructed from a Hermann-Mauguin symbol or an ITA
number. We start with NaCl, space group Fm-3m (SG 225).

In [2]:
sg_nacl = SpaceGroup("Fm-3m")
print(f"Number:         {sg_nacl.number}")
print(f"Symbol:         {sg_nacl.symbol}")
print(f"Crystal system: {sg_nacl.crystal_system}")
print(f"Centering:      {sg_nacl.centering_type}")
print(f"Point group:    {sg_nacl.point_group.symbol}")

Number:         225
Symbol:         Fm-3m
Crystal system: cubic
Centering:      F
Point group:    m-3m


For space groups with multiple origin choices, a setting suffix selects the
variant. Silicon's space group Fd-3m (SG 227) has two origin choices; the
ITA default is origin choice 2.

In [3]:
sg_si = SpaceGroup("Fd-3m")          # origin choice 2 by default
sg_si_oc1 = SpaceGroup("Fd-3m", setting="1")  # origin choice 1
print(f"Fd-3m (default):   {sg_si.xhm_symbol}")
print(f"Fd-3m (choice 1):  {sg_si_oc1.xhm_symbol}")
print(f"Operator count:    {len(sg_si.operators)} (48 proper + centering)")

Fd-3m (default):   F d -3 m :2
Fd-3m (choice 1):  F d -3 m :1
Operator count:    48 (48 proper + centering)


## NaCl: loading a crystal and expanding the asymmetric unit

`Crystal.from_cif()` reads the lattice parameters, space group symbol, and
asymmetric unit from a CIF file. The asymmetric unit for NaCl contains only
two sites: one Na and one Cl.

In [4]:
nacl = Crystal.from_cif("nacl.cif")
print(f"Space group from CIF: {nacl.space_group}")
print(f"Lattice: a = {nacl.a} Å")
print()
print("Asymmetric unit:")
for label, site in nacl.sites.items():
    print(f"  {label}: {site.ion} at {site.position}")

Space group from CIF: F m -3 m
Lattice: a = 5.62 Å

Asymmetric unit:
  Na1: Na1+ at [0. 0. 0.]
  Cl1: Cl1- at [0.5 0.5 0.5]


`Crystal.expand_sites()` applies every space group operator and centering
vector to each asymmetric unit site, wraps coordinates to [0, 1), and
removes duplicates.

In [5]:
full_nacl = nacl.expand_sites(sg_nacl)
print(f"Full unit cell: {len(full_nacl)} sites")
print()
print(f"{'Ion':>6}  {'x':>7}  {'y':>7}  {'z':>7}")
for site in full_nacl:
    x, y, z = site.position
    print(f"{site.ion:>6}  {x:>7.4f}  {y:>7.4f}  {z:>7.4f}")

Full unit cell: 8 sites

   Ion        x        y        z
  Na1+   0.0000   0.0000   0.0000
  Na1+   0.0000   0.5000   0.5000
  Na1+   0.5000   0.0000   0.5000
  Na1+   0.5000   0.5000   0.0000
  Cl1-   0.5000   0.5000   0.5000
  Cl1-   0.5000   0.0000   0.0000
  Cl1-   0.0000   0.5000   0.0000
  Cl1-   0.0000   0.0000   0.5000


## Silicon (Fd-3m): diamond structure

Silicon has the diamond structure: F-centering plus a d-glide, which
produces absences beyond those of a simple face-centred lattice. One Si
site in the asymmetric unit expands to 8 atoms in the unit cell.

In [6]:
si = Crystal.from_cif("silicon.cif")
print(f"Space group: {si.space_group}, a = {si.a:.5f} Å")
print()
print("Asymmetric unit:")
for label, site in si.sites.items():
    print(f"  {label}: {site.ion} at {site.position}")

Space group: F d -3 m, a = 5.43070 Å

Asymmetric unit:
  Si1: Si at [0.125 0.125 0.125]


In [7]:
full_si = si.expand_sites(sg_si)
print(f"Full unit cell: {len(full_si)} Si atoms")
print()
print(f"{'x':>8}  {'y':>8}  {'z':>8}")
for site in full_si:
    x, y, z = site.position
    print(f"{x:>8.4f}  {y:>8.4f}  {z:>8.4f}")

Full unit cell: 8 Si atoms

       x         y         z
  0.1250    0.1250    0.1250
  0.1250    0.6250    0.6250
  0.6250    0.1250    0.6250
  0.6250    0.6250    0.1250
  0.8750    0.3750    0.3750
  0.8750    0.8750    0.8750
  0.3750    0.3750    0.8750
  0.3750    0.8750    0.3750


## Systematic absences

`SpaceGroup.is_systematically_absent()` applies the h·W = h criterion
with exact rational arithmetic. We compare the allowed reflections for
Fm-3m (NaCl) and Fd-3m (Si) to see what the d-glide adds.

In [ ]:
# Low-index reflections to check
reflections = [
    (1, 0, 0), (1, 1, 0), (1, 1, 1),
    (2, 0, 0), (2, 1, 0), (2, 2, 0),
    (2, 2, 2), (4, 0, 0),
]

print(f"{'hkl':>8}  {'Fm-3m (NaCl)':>14}  {'Fd-3m (Si)':>12}")
print("-" * 40)
for hkl in reflections:
    nacl_status = "absent" if sg_nacl.is_systematically_absent(hkl) else "allowed"
    si_status = "absent" if sg_si.is_systematically_absent(hkl) else "allowed"
    print(f"{hkl!r:>8}  {nacl_status:>14}  {si_status:>12}")

Both Fm-3m and Fd-3m impose F-centering conditions (h, k, l must be all odd
or all even). Fd-3m adds d-glide absences: the (200) reflection, allowed in
Fm-3m, is absent in Fd-3m.

## Corundum (R-3c, trigonal)

Corundum (Al₂O₃) has rhombohedral R-centering and a c-glide. In the
hexagonal setting, the unit cell contains 6 formula units (12 Al + 18 O).
Two sites in the asymmetric unit expand to 30 atoms.

In [11]:
cor = Crystal.from_cif("corundum.cif")
sg_cor = SpaceGroup("R-3c")

print(f"Space group: {cor.space_group}, number {sg_cor.number}")
print(f"Crystal system: {sg_cor.crystal_system}, centering: {sg_cor.centering_type}")
print(f"a = {cor.a:.4f} Å,  c = {cor.c:.4f} Å,  γ = {cor.gamma:.1f}°")
print()
print("Asymmetric unit:")
for label, site in cor.sites.items():
    x, y, z = site.position
    print(f"  {label}: {site.ion} at ({x:.5f}, {y:.5f}, {z:.5f})")

Space group: R -3 c, number 167
Crystal system: trigonal, centering: R
a = 4.7602 Å,  c = 12.9933 Å,  γ = 120.0°

Asymmetric unit:
  Al1: Al at (0.00000, 0.00000, 0.35228)
  O1: O at (0.30624, 0.00000, 0.25000)


In [ ]:
full_cor = cor.expand_sites(sg_cor)
al_sites = [s for s in full_cor if "Al" in s.ion]
o_sites = [s for s in full_cor if "O" in s.ion]
print(f"Full unit cell: {len(al_sites)} Al + {len(o_sites)} O = {len(full_cor)} atoms")
print("(Z = 6 formula units of Al2O3)")

## Fractional and Cartesian coordinates

`DirectLattice.fractional_to_cartesian()` converts fractional coordinates
to Cartesian using the ITC Vol B orthogonalization matrix with a parallel to
x and b in the xy-plane. The inverse converts back.

In [13]:
print("NaCl: full unit cell positions in Cartesian (Å)")
print(f"{'Ion':>6}  {'x (Å)':>8}  {'y (Å)':>8}  {'z (Å)':>8}")
for site in full_nacl:
    cart = nacl.lattice.fractional_to_cartesian(site.position)
    print(f"{site.ion:>6}  {cart[0]:>8.4f}  {cart[1]:>8.4f}  {cart[2]:>8.4f}")

NaCl: full unit cell positions in Cartesian (Å)
   Ion     x (Å)     y (Å)     z (Å)
  Na1+    0.0000    0.0000    0.0000
  Na1+    0.0000    2.8100    2.8100
  Na1+    2.8100    0.0000    2.8100
  Na1+    2.8100    2.8100    0.0000
  Cl1-    2.8100    2.8100    2.8100
  Cl1-    2.8100    0.0000    0.0000
  Cl1-    0.0000    2.8100    0.0000
  Cl1-    0.0000    0.0000    2.8100


In [14]:
# Round-trip: fractional -> Cartesian -> fractional
frac_in = np.array([0.5, 0.5, 0.5])
cart = nacl.lattice.fractional_to_cartesian(frac_in)
frac_out = nacl.lattice.cartesian_to_fractional(cart)
print(f"Fractional in:  {frac_in}")
print(f"Cartesian:      {cart} Å")
print(f"Fractional out: {np.round(frac_out, 10)}")

Fractional in:  [0.5 0.5 0.5]
Cartesian:      [2.81 2.81 2.81] Å
Fractional out: [0.5 0.5 0.5]


## Allowed reflections table

We enumerate low-index reflections and filter out systematic absences to
produce a list of observable peaks for each structure. This is the first
step toward computing diffraction intensities.

In [ ]:
def allowed_reflections(sg, hkl_max=3):
    """Return all (h, k, l) with 0 <= h <= hkl_max that are not absent."""
    allowed = []
    for h in range(0, hkl_max + 1):
        for k in range(-hkl_max, hkl_max + 1):
            for ll in range(-hkl_max, hkl_max + 1):
                if h == 0 and k == 0 and ll == 0:
                    continue
                hkl = (h, k, ll)
                if not sg.is_systematically_absent(hkl):
                    allowed.append(hkl)
    return allowed


structures = [
    ("Fm-3m (NaCl)", sg_nacl),
    ("Fd-3m (Si)", sg_si),
    ("R-3c (corundum)", sg_cor),
]
for name, sg in structures:
    n = len(allowed_reflections(sg))
    print(f"{name}: {n} allowed reflections with h in [0,3]")

The next notebook introduces structure factors, which weight each allowed
reflection by the atomic scattering amplitudes to predict diffraction
intensities.